In [ ]:
import numpy as np
import pandas as pd
from torch import nn
import torch
import torch.utils.data as data
import matplotlib.pyplot as plt
import warnings

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi']=100

device="cuda" if torch.cuda.is_available() else "cpu"
warnings.filterwarnings('ignore')

In [ ]:
class MyDataset(data.Dataset):
    def __init__(self,data_list):
        self.data_list=data_list
    def __len__(self):
        return len(self.data_list)
    def __getitem__(self,index):
        x=self.data_list[index][0]
        y=self.data_list[index][1]
        return x,y
    

In [ ]:

def train(dataloader,model,loss_fn,optimizer):
    size=len(dataloader.dataset)
    num_batches=len(dataloader)

    train_loss,train_acc=0,0

    for imgs,target in dataloader:
        imgs,target=imgs.to(device),target.to(device)

        imgs=imgs.to(device,dtype=torch.float)
        
        
        pred=model(imgs)
        target=target.long()
        
        
        loss=loss_fn(pred,target)

        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        
        train_acc+=(pred.argmax(1)==target).type(torch.float).sum().item()
        train_loss+=loss.item()

    train_acc/=size
    train_loss/=num_batches

    return train_acc,train_loss


def val(dataloader,model,loss_fn):
    size=len(dataloader.dataset)
    num_batches=len(dataloader)
    val_loss,val_acc=0,0

    
    with torch.no_grad():
        for imgs,target in dataloader:
            imgs,target=imgs.to(device),target.to(device)

            imgs=imgs.to(device,dtype=torch.float)
            
            
            #计算loss
            target_pred=model(imgs)
            target=target.long()
            loss=loss_fn(target_pred,target)

            val_loss+=loss.item()
            val_acc+=(target_pred.argmax(1)==target).type(torch.float).sum().item()

    val_acc/=size
    val_loss/=num_batches
    return val_acc,val_loss

In [ ]:
data_=pd.read_excel(r"D:\20250106project\2.comebine\CNN.xlsx",index_col=0)
X=data_.iloc[:,:-1].values
y=data_.iloc[:,-1].values


In [ ]:
columns = data_.iloc[:,:-1].columns.values

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(X, y,train_size=0.7, random_state=42 ,stratify=y)


X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=2/3, random_state=28,stratify=y_temp)

print("训练集特征形状:", X_train.shape)
print("训练集标签形状:", y_train.shape)
print("验证集特征形状:", X_val.shape)
print("验证集标签形状:", y_val.shape)
print("测试集特征形状:", X_test.shape)
print("测试集标签形状:", y_test.shape)

In [ ]:
X_train=X_train.reshape((-1,1,16001))
X_test=X_test.reshape((-1,1,16001))
X_val=X_val.reshape((-1,1,16001))
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

In [ ]:
train_data=[]
test_data=[]
val_data=[]

train_data=[[i,j] for i,j in zip(X_train,y_train)]
test_data=[[i,j] for i,j in zip(X_test,y_test)]
val_data=[[i,j] for i,j in zip(X_val,y_val)]

In [ ]:
batch=64

train_dataset=MyDataset(train_data)
train_dataloader=data.DataLoader(train_dataset,batch_size=batch,shuffle=True)

test_dataset=MyDataset(test_data)
test_dataloader=data.DataLoader(test_dataset,batch_size=batch,shuffle=False)

val_dataset=MyDataset(val_data)
val_dataloader=data.DataLoader(val_dataset,batch_size=batch,shuffle=False)
  
print('Train set: {} samples'.format(len(train_dataloader)))  
print('Test set: {} samples'.format(len(test_dataloader)))  
print('val set: {} samples'.format(len(val_dataloader)))  

  
images, labels = next(iter(train_dataloader))  
print('Image batch shape:', images.size())  

In [ ]:
class OneDCNN(nn.Module):
    def __init__(self):
        super(OneDCNN, self).__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=3)  
        self.bn1 = nn.BatchNorm1d(32)  
        self.relu1 = nn.ReLU()  

        self.maxpool1 = nn.MaxPool1d(2)  

        self.conv2 = nn.Conv1d(32, 64, kernel_size=3)  
        self.bn2 = nn.BatchNorm1d(64)  
        self.relu2 = nn.ReLU()  

        self.maxpool2 = nn.MaxPool1d(2)  

        
        self.fc4 = None  

        self.dropout = nn.Dropout(0.2)  

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.maxpool1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        x = self.maxpool2(x)

        
        print(f"After conv1 and maxpool1: {x.shape}")
        print(f"After conv2 and maxpool2: {x.shape}")

       
        flat_size = x.view(x.size(0), -1).size(1)

        
        if self.fc4 is None:
            self.fc4 = nn.Linear(flat_size, 50)  

        x = x.view(x.size(0), -1)
        print(f"After flatten: {x.shape}")

        x = self.fc4(x)
        x = self.dropout(x)

        return x

In [ ]:
model = OneDCNN()

In [ ]:
import torchsummary as summary
summary.summary(model,(1,16001))

In [ ]:

import copy
import time

optimizer=torch.optim.Adam(model.parameters(),1e-4)
# optimizer=torch.optim.SGD(model1.parameters(),lr=1e-4,momentum=1)
loss_fn=nn.CrossEntropyLoss()

epochs=100

train_loss=[]
train_acc=[]
val_loss=[]
val_acc=[]

best_acc=0


start_time=time.time()

for epoch in range(epochs):
    model.train()
    _,_=train(train_dataloader,model,loss_fn,optimizer)

    model.eval()
    epoch_train_acc,epoch_train_loss=val(train_dataloader,model,loss_fn)
    epoch_val_acc,epoch_val_loss=val(val_dataloader,model,loss_fn)

    
    if epoch_val_acc>best_acc:
        best_acc=epoch_val_acc
        best_model=copy.deepcopy(model)
        print('最佳模型epoch为:',epoch)

    train_acc.append(epoch_train_acc)
    train_loss.append(epoch_train_loss)
    val_acc.append(epoch_val_acc)
    val_loss.append(epoch_val_loss)

    
    lr=optimizer.state_dict()['param_groups'][0]['lr']
    template=('Epoch:{:2d},Train_acc:{:.1f}%,Train_loss:{:.3f},Val_acc:{:.1f}%,Val_loss:{:.3f},Lr:{:.4f}')
    print(template.format(epoch+1,epoch_train_acc*100,epoch_train_loss,epoch_val_acc*100,epoch_val_loss,lr))



end_time=time.time()
execution_time=end_time-start_time
print("执行时间：",execution_time,"秒")

print('Done')

In [ ]:
epochs_range=range(len(train_loss))
plt.rcParams['font.family'] = ['SimHei']
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)

plt.plot(epochs_range,train_acc,label='Training Accuracy')
plt.plot(epochs_range,val_acc,label='val Accuracy')
plt.legend(loc='best')
plt.xlabel('迭代次数')
plt.ylabel('accuracy')
plt.title('Training and val Accuracy')
plt.ylim(0, 1.0)

plt.subplot(1,2,2)
plt.plot(epochs_range,train_loss,label='Training Loss')
plt.plot(epochs_range,val_loss,label='val Loss')
plt.legend(loc='best')
plt.xlabel('迭代次数')
plt.ylabel('loss')
plt.title('Training and val Loss')
plt.savefig('accuracy_loss.png',dpi=800)

plt.show()

In [ ]:

PATH='./oneCNN_model.pth'
torch.save(best_model.state_dict(),PATH)

In [ ]:
X_test_torch=torch.from_numpy(X_test)
X_test_torch=torch.reshape(X_test_torch,(-1,1,16001))
X_test_torch=X_test_torch.to(dtype=torch.float)
model.eval()
with torch.no_grad():
    outputs_LeNet = model(X_test_torch)
print(outputs_LeNet)

In [ ]:
def SCIPlot():
    plt.grid(False)
    plt.box(True)
    ax = plt.gca()
    for spine in ax.spines.values():
        spine.set_linewidth(2)
    ax.tick_params(axis='both', which='major', labelsize=16)
    ax.set_facecolor('w')
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)
    plt.rcParams['font.family'] = ['Times New Roman', 'Arial', 'Helvetica']
    return ax

In [ ]:
from sklearn.metrics import roc_curve,auc
y_pred_proba_smica=outputs_LeNet

fpr_smica=dict()
tpr_smica=dict()
roc_auc_smica=dict()
classes=[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49]
for i in range(50):
    
    fpr_smica[i],tpr_smica[i],_=roc_curve(y_test,y_pred_proba_smica[:,i],pos_label=classes[i])
    roc_auc_smica[i]=auc(fpr_smica[i],tpr_smica[i])   


from sklearn.preprocessing import label_binarize
y_test_bi=label_binarize(y_test, classes=[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49])

fpr_smica["micro"], tpr_smica["micro"], _ = roc_curve(y_test_bi.ravel(), y_pred_proba_smica.ravel())
roc_auc_smica["micro"] = auc(fpr_smica["micro"], tpr_smica["micro"])

In [ ]:
fig, ax = plt.subplots() 
# Plot all ROC curves
ax.plot(fpr_smica["micro"], tpr_smica["micro"],
         label='CNN ROC curve (auc = {0:0.2f})'
               ''.format(roc_auc_smica["micro"]),
         color='plum', linestyle='-', linewidth=1)

plt.plot([-0.05, 1.05], [-0.05, 1.05], 'k--', lw=1)

plt.xlim([-0.05, 1.04])
plt.ylim([-0.05, 1.04])


plt.xlabel('False Positive Rate',fontsize=14)
plt.ylabel('True Positive Rate',fontsize=14)
leg=plt.legend(loc="lower right")
leg.legendPatch.set_linewidth(1.5)
leg.get_frame().set_edgecolor('black')
plt.tick_params(axis='x', which='both', size=5, labelsize=14)  
plt.tick_params(axis='y', which='both', size=5, labelsize=14) 

plt.tick_params(top='on', right='on', which='both') 
plt.rcParams['xtick.direction'] = 'in' 
plt.rcParams['ytick.direction'] = 'in' 

SCIPlot()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import scikitplot as skplt
y_pred_test = y_pred_proba_smica
print(y_pred_test)
skplt.metrics.plot_roc_curve(y_test, y_pred_test)
plt.gcf().set_size_inches(15, 15)

plt.title("ROC Curve", fontsize=30)  
plt.xlabel("False Positive Rate", fontsize=24)  
plt.ylabel("True Positive Rate", fontsize=24)  


plt.xticks(fontsize=15)  
plt.yticks(fontsize=15)  

plt.legend(fontsize=12, loc='center left', bbox_to_anchor=(1, 0.5))  

plt.tight_layout()  

plt.show()  

输出各种指标

In [ ]:

from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score
from sklearn.metrics import classification_report

In [ ]:
y_pred_test_p=y_pred_test.argmax(1)
np.array(y_pred_test_p)

In [ ]:
X_train_torch=torch.from_numpy(X_train)
X_train_torch=torch.reshape(X_train_torch,(-1,1,16001))
X_train_torch=X_train_torch.to(dtype=torch.float)
model.eval()
with torch.no_grad():
    outputs_train_LeNet = best_model(X_train_torch)
print(outputs_train_LeNet)

In [ ]:
y_pred_train_p=outputs_train_LeNet.argmax(1)
y_pred_train_p

In [ ]:
accuracy_test=accuracy_score(y_train, y_pred_train_p)
precision_test = precision_score(y_train, y_pred_train_p,average='macro')
recall_test = recall_score(y_train, y_pred_train_p,average='macro')
f1_test = f1_score(y_train, y_pred_train_p,average='macro')
confumatrix_test=confusion_matrix(y_train, y_pred_train_p)
classreport_test=classification_report(y_train, y_pred_train_p)
print("训练集准确度：",accuracy_test)
print("训练集精确度：",precision_test)
print("训练集召回率：",recall_test)
print("训练集f1分数：",f1_test)
print("训练集的混淆矩阵：\n",confumatrix_test)
print("训练集的分类报告：\n",classreport_test)
print("*"*100)

In [ ]:
mapping = { 
    0: 'LIN',
    1: 'CAS',
    2: 'CAN',
    3: 'SSG', 
    4: 'AES',
    5: 'IMP',
    6: 'ORO',
    7: 'MOM',
    8: 'PLS',
    9: 'JUG',
    10: 'PER',
    11: 'TOR',
    12: 'ENT',
    13: 'ARE',
    14: 'CIT',
    15: 'ACS',
    16: 'SSP',
    17: 'TST',
    18: 'AST',
    19: 'PHA',
    20: 'VAC',
    21: 'TRS',
    22: 'MEL',
    23: 'LAB',
    24: 'GIN',
    25: 'MYR',
    26: 'STE',
    27: 'TRI',
    28: 'EUR',
    29: 'SIN',
    30: 'ABU',
    31: 'ARM',
    32: 'ALP',
    33: 'LIT',
    34: 'RAP',
    35: 'NES',
    36: 'NEP',
    37: 'DES',
    38: 'RIC',
    39: 'COI',
    40: 'VIG',
    41: 'PLA',
    42: 'PRU',
    43: 'ZIZ',
    44: 'CEL',
    45: 'ALL',
    46: 'NIG',
    47: 'SES',
    48: 'SSN',
    49: 'LON',

    
} 

y_train_= [mapping[i] if i in mapping else i for i in y_train] 
y_pred_train_ = [mapping[i] if i in mapping else i for i in np.array(y_pred_train_p)] 
plot = skplt.metrics.plot_confusion_matrix(y_train_, y_pred_train_,x_tick_rotation=45,text_fontsize=12,figsize=(18,18))
# 设置标题和轴标题的字体大小
plt.title('1D-CNN-TRAIN', fontsize=30)  # 设置标题的字体大小
plt.xlabel('True Labels', fontsize=24)  # 设置X轴标题的字体大小
plt.ylabel('Predicted Labels', fontsize=24)  # 设置Y轴标题的字体大小
plt.tick_params(axis='both', which='both', length=0)  
plt.savefig('random forest all.png', dpi=800,bbox_inches='tight') 
plt.grid(False)
plt.show()

In [ ]:
accuracy_test=accuracy_score(y_test, y_pred_test_p)
precision_test = precision_score(y_test, y_pred_test_p,average='macro')
recall_test = recall_score(y_test, y_pred_test_p,average='macro')
f1_test = f1_score(y_test, y_pred_test_p,average='macro')
confumatrix_test=confusion_matrix(y_test, y_pred_test_p)
classreport_test=classification_report(y_test, y_pred_test_p)
print("测试集准确度：",accuracy_test)
print("测试集精确度：",precision_test)
print("测试集召回率：",recall_test)
print("测试集f1分数：",f1_test)
print("测试集的混淆矩阵：\n",confumatrix_test)
print("测试集的分类报告：\n",classreport_test)
print("*"*100)

绘制混淆矩阵

In [ ]:
mapping = { 
    0: 'LIN',
    1: 'CAS',
    2: 'CAN',
    3: 'SSG', 
    4: 'AES',
    5: 'IMP',
    6: 'ORO',
    7: 'MOM',
    8: 'PLS',
    9: 'JUG',
    10: 'PER',
    11: 'TOR',
    12: 'ENT',
    13: 'ARE',
    14: 'CIT',
    15: 'ACS',
    16: 'SSP',
    17: 'TST',
    18: 'AST',
    19: 'PHA',
    20: 'VAC',
    21: 'TRS',
    22: 'MEL',
    23: 'LAB',
    24: 'GIN',
    25: 'MYR',
    26: 'STE',
    27: 'TRI',
    28: 'EUR',
    29: 'SIN',
    30: 'ABU',
    31: 'ARM',
    32: 'ALP',
    33: 'LIT',
    34: 'RAP',
    35: 'NES',
    36: 'NEP',
    37: 'DES',
    38: 'RIC',
    39: 'COI',
    40: 'VIG',
    41: 'PLA',
    42: 'PRU',
    43: 'ZIZ',
    44: 'CEL',
    45: 'ALL',
    46: 'NIG',
    47: 'SES',
    48: 'SSN',
    49: 'LON',


    
} 


y_test_= [mapping[i] if i in mapping else i for i in y_test] 
y_pred_test_ = [mapping[i] if i in mapping else i for i in np.array(y_pred_test_p)] 
plot = skplt.metrics.plot_confusion_matrix(y_test_, y_pred_test_,x_tick_rotation=45,text_fontsize=12,figsize=(20,20))
# 设置标题和轴标题的字体大小
plt.title('1D-CNN-TEXT', fontsize=28)  # 设置标题的字体大小
plt.xlabel('True Labels', fontsize=24)  # 设置X轴标题的字体大小
plt.ylabel('Predicted Labels', fontsize=24)  # 设置Y轴标题的字体大小
plt.tick_params(axis='both', which='both', length=0)  
plt.savefig('random forest all.png', dpi=800,bbox_inches='tight') 
plt.grid(False)
plt.show()

In [ ]:
data=pd.read_excel(r'C:\Users\唐梓竣\Desktop\20241212test\result.xlsx')

In [ ]:
X_outer=data.iloc[:,1:-1].values
y_outer=data.iloc[:,-1].values

In [ ]:
X_outer_torch=torch.from_numpy(X_outer)
X_outer_torch=torch.reshape(X_outer_torch,(-1,1,16001))
X_outer_torch=X_outer_torch.to(dtype=torch.float)
model.eval()
with torch.no_grad():
    outputs_outer = best_model(X_outer_torch)
print(outputs_outer)
outputs_outer.argmax(1)

In [ ]:
y_outer

In [ ]:

X_train_=pd.DataFrame(X_train.reshape(X_train.shape[0],-1),columns=columns)
X_train_.head()

In [ ]:
tensor_X_train = torch.tensor(X_train,dtype=torch.float32)
tensor_X_train.shape